# Notebook 02: Predict

# Outline 

1. Import Libraries and Functions
2. Intake Input Data
3. Validate Input Data
4. Clean Input Data 
5. Configure Run
6. Import Models 
7. Generate Predictions

# 1. Import Libraries and Functions

In [0]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import pandas as pd

# sections 2, 6 and 7
from classifier import (
    ID_LEVELS, NAME_LEVELS, DELIMITER, OUTPUT_DIR,
    predictions_path, timestamp,
)

# sections 2 and 3
from classifier.predict import read_input, from_records, validate

# sections 5 and 7
from classifier.predict import load_thresholds, predict, summarise

# section 6
from classifier import artifacts

# 2. Intake Input Data

In [0]:
# # Pipe-delimited, first column the product identifier, second the text
# # description. Comment out whichever options you are not using; the last
# # uncommented assignment is the one that applies.

# # single product
# records = from_records(("V-0001", "BRASS BALL VALVE 1 IN NPT THREADED FULL PORT"))

# # list of products
# records = from_records([
#     ("V-0001", "BRASS BALL VALVE 1 IN NPT THREADED FULL PORT"),
#     ("V-0002", "NITRILE EXAM GLOVE PWDR FREE BLU MED 100/BX"),
#     ("V-0003", "QJX8A TORQUE HEAD BOXEND 1/4 12 PT"),
#     ("V-0004", "ZZQQ WXYZZY PLMNOP"),
# ])

# load from file. The identifier and description are taken positionally
# from the first two columns, so select them by name if the file carries
# any others: passing the frame through unselected classifies whatever
# column happens to sit second. Extra columns are ignored either way.
records = read_input(r"data/to_predict.txt", delimiter=DELIMITER)[
    ["Vallen ID", "FullDesc"]
]

records

,Vallen ID,FullDesc
0,781568M3545,M3545 LEESON C6T34FB1C CAT1101 3450/2850RPM 1HP
1,16507834,28997938 MOTOR ELECTRIC AC 480VAC 3000 RPM SERVO
2,10882710,1MB21212AA114AG3 MOTOR 7.5HP 3520 RPM 213T TFEC
3,66244100PPG000007202A,MOTOR ELECTRIC AC 3HP 460V 1140 RPM
4,BAL EM3774T,MOTOR 3PH 10HP 1760RPM TEFC 575V
...,...,...
144995,406442PB3873,PB3873 POLY BAG 6X9 6MIL RECLOSEABLE CA/1000
144996,422212CZ20203,CZ20203 BAG ZIPLOCK 2X3 2MIL
144997,4343078X84MIL,8INX8IN 4MIL ZIPLOC 1000/CA
144998,14947405,901-2490-17 ZIPLOCK BAGS W/EYELET 15X12 AND


# 3. Validate Input Data

In [0]:
# Raises on an empty input, fewer than two columns, a blank identifier or
# a duplicate identifier — a duplicate would make the output impossible to
# join back, so it stops the run rather than being silently deduplicated.
ID_COL_IN, DESC_COL_IN = validate(records)

[input] 145,000 rows | id column 'Vallen ID' | description column 'FullDesc'


# 4. Clean Input Data

In [0]:
# The cleaning recipe is loaded with the model in section 6 and applied
# inside predict() in section 7, so the text is cleaned exactly as the
# training text was. Nothing to configure here.
print("cleaning is applied by predict(), using the recipe saved with the model")

cleaning is applied by predict(), using the recipe saved with the model


# 5. Configure Run

In [0]:
# Report every level, or Level 4 only. Unrequested levels are still
# reported as columns, with 'not requested' as the reason, so the output
# shape never changes.
ALL_LEVELS = True

# Apply per-level confidence thresholds, refusing a level whose confidence
# falls short. Refusing a level also refuses everything under it.
#
# The values in config/thresholds.json predate the change to reporting
# scores as the models produce them, so they are on the wrong footing and
# would refuse nearly everything. Refit them against these scores before
# switching this on.
USE_THRESHOLDS = False # True 

# Out-of-vocabulary items are always refused, at every level, whatever
# their confidence — the model would be answering from category
# frequencies rather than from the product.

thresholds = load_thresholds() if USE_THRESHOLDS else None
print("thresholds:", thresholds if thresholds else "not applied")

thresholds: not applied


# 6. Import Models

In [0]:
MODEL_DIR = artifacts.latest_model_dir(OUTPUT_DIR)

clf, cleaner, lookup, manifest = artifacts.load(MODEL_DIR)

print(f"model      : {os.path.basename(MODEL_DIR)}")
print(f"trained on : {manifest.get('n_training_rows', 0):,} rows")
print(f"vocabulary : {manifest.get('n_features', 0):,} features")
print(f"categories : {manifest.get('n_paths', 0):,} paths")
print(f"lookup     : {'loaded' if lookup is not None else 'missing — IDs only'}")

model      : model_20260915_193002
trained on : 456,537 rows
vocabulary : 60,000 features
categories : 3,646 paths
lookup     : loaded


# 7. Generate Predictions

In [0]:
predictions = predict(
    clf, records, cleaner,
    lookup=lookup,
    all_levels=ALL_LEVELS,
    thresholds=thresholds,
)

predictions


[predict] 145,000 rows
[predict] 1,348 refused as out of vocabulary
[predict] L1: 143,652 predicted, 1,348 refused
[predict] L2: 143,652 predicted, 1,348 refused
[predict] L3: 143,652 predicted, 1,348 refused
[predict] L4: 143,652 predicted, 1,348 refused


,Vallen ID,Predicted Level 1 ID,Predicted Level 2 ID,Predicted Level 3 ID,Predicted Level 4 ID,Predicted Level 1 Name,Predicted Level 2 Name,Predicted Level 3 Name,Predicted Level 4 Name,L1 Confidence,L2 Confidence,L3 Confidence,L4 Confidence,L1 Refusal Reason,L2 Refusal Reason,L3 Refusal Reason,L4 Refusal Reason
0,781568M3545,L1-12810533,L2-12810536,L3-12810573,L4-12810800,Tools & Equipment,Power Tools,Grinding Tools & Accessories,Die Grinders - Corded,0.006746,0.005966,0.005653,0.003461,,,,
1,16507834,L1-12785504,L2-12785509,L3-12785530,L4-12785646,Bearings & Power Transmission,Motors & Motor Components,Motors,AC Motors,0.999438,0.999438,0.999438,0.992449,,,,
2,10882710,L1-12785504,L2-12785509,L3-12785530,L4-12785646,Bearings & Power Transmission,Motors & Motor Components,Motors,AC Motors,0.992067,0.992067,0.992067,0.899808,,,,
3,66244100PPG000007202A,L1-12785504,L2-12785509,L3-12785530,L4-12785646,Bearings & Power Transmission,Motors & Motor Components,Motors,AC Motors,0.999769,0.999769,0.999769,0.999628,,,,
4,BAL EM3774T,L1-12785504,L2-12785509,L3-12785530,L4-12785646,Bearings & Power Transmission,Motors & Motor Components,Motors,AC Motors,0.999997,0.999997,0.999997,0.999997,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144995,406442PB3873,L1-12810458,L2-12810462,L3-12810473,L4-12810502,Shipping & Packaging,Shipping Supplies,Bags,Bags - Flat,0.885323,0.885323,0.885323,0.740968,,,,
144996,422212CZ20203,L1-12810458,L2-12810462,L3-12810473,L4-12810502,Shipping & Packaging,Shipping Supplies,Bags,Bags - Flat,0.669289,0.669289,0.669289,0.623032,,,,
144997,4343078X84MIL,L1-12810985,L2-12810996,L3-12811034,L4-13322444,Workplace Supplies,Food & Beverage Serving,Serveware & Food Storage,Zip-Top Bags,0.195520,0.195520,0.195520,0.195447,,,,
144998,14947405,L1-12810985,L2-12810996,L3-12811034,L4-13322444,Workplace Supplies,Food & Beverage Serving,Serveware & Food Storage,Zip-Top Bags,0.444610,0.444610,0.444610,0.444587,,,,


In [0]:
summarise(predictions)

,level,predicted,out of vocabulary,below threshold,no parent prediction,not requested,share_predicted
0,L1,143652,1348,0,0,0,0.990703
1,L2,143652,1348,0,0,0,0.990703
2,L3,143652,1348,0,0,0,0.990703
3,L4,143652,1348,0,0,0,0.990703


In [0]:
STAMP = timestamp()
OUT_PATH = predictions_path(STAMP)

predictions.to_csv(OUT_PATH, index=False, sep=DELIMITER)
print(f"wrote {len(predictions):,} rows to {OUT_PATH}")
print(f"columns: {list(predictions.columns)}")

wrote 145,000 rows to /Workspace/Users/patrick.white@vallen.com/Category Classification/NB Category Classifier Minimized/outputs/predictions_20260915_195504.txt
columns: ['Vallen ID', 'Predicted Level 1 ID', 'Predicted Level 2 ID', 'Predicted Level 3 ID', 'Predicted Level 4 ID', 'Predicted Level 1 Name', 'Predicted Level 2 Name', 'Predicted Level 3 Name', 'Predicted Level 4 Name', 'L1 Confidence', 'L2 Confidence', 'L3 Confidence', 'L4 Confidence', 'L1 Refusal Reason', 'L2 Refusal Reason', 'L3 Refusal Reason', 'L4 Refusal Reason']
